In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
import mediapy

from motrixsim import SceneData, load_model, step
from gaussian_renderer import GSRendererMotrixSim
from gs_playground import ROOT_PATH

mjcf_path = ROOT_PATH / "models" / "robots" / "manipulation" / "franka_robotiq" / "xmls" / "pick_fruit.xml"

_ASSETS_FRANKA_DIR = ROOT_PATH / "models" / "robots" / "manipulation" / "franka_robotiq"
_ASSETS_BANANA_DIR = ROOT_PATH / "models" / "tasks" / "table30" / "03_arrange_fruits_in_basket"

gaussians = {
    "link0" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link0.ply").as_posix(),
    "link1" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link1.ply").as_posix(),
    "link2" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link2.ply").as_posix(),
    "link3" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link3.ply").as_posix(),
    "link4" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link4.ply").as_posix(),
    "link5" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link5.ply").as_posix(),
    "link6" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link6.ply").as_posix(),
    "link7" : (_ASSETS_FRANKA_DIR / "3dgs" / "franka" / "link7.ply").as_posix(),

    "robotiq_base"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "robotiq_base.ply").as_posix(),
    "left_driver"       : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_driver.ply").as_posix(),
    "left_coupler"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_coupler.ply").as_posix(),
    "left_spring_link"  : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_spring_link.ply").as_posix(),
    "left_follower"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "left_follower.ply").as_posix(),

    "right_driver"      : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_driver.ply").as_posix(),
    "right_coupler"     : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_coupler.ply").as_posix(),
    "right_spring_link" : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_spring_link.ply").as_posix(),
    "right_follower"    : (_ASSETS_FRANKA_DIR / "3dgs" / "robotiq" / "right_follower.ply").as_posix(),

    "banana"            : (_ASSETS_BANANA_DIR / "3dgs" / "fruit_banana.ply").as_posix()
}
background = (_ASSETS_FRANKA_DIR / "3dgs" / "background.ply").as_posix()

## Single Env

In [4]:
# Load the scene model
mx_model = load_model(mjcf_path.as_posix())
mx_data = SceneData(mx_model)

# Initialize the simulation
step(mx_model, mx_data)

gaussians.update({"background": background})
gsmx_renderer = GSRendererMotrixSim(gaussians, mx_model)
gaussians.pop("background")
gsmx_renderer.update_gaussians(mx_data)
results = gsmx_renderer.render(mx_model, mx_data, list(range(len(mx_model.cameras))), 320, 240)
for k, (rgb, depth) in results.items():
    rgb_np = rgb.cpu().numpy()
    mediapy.show_image(rgb_np)

RuntimeError: failed to load scene, read_to_buf failed: /home/tatp/code/gs_playground/gs_playground/models/robots/manipulation/franka_robotiq/xmls/pick_fruit.xml: No such file or directory (os error 2)

## Batch Env

In [ ]:
from gaussian_renderer import BatchSplatConfig, MtxBatchSplatRenderer

num_env = 16

# Load the scene model
mx_model = load_model(mjcf_path.as_posix())
mx_data = SceneData(mx_model, batch=(num_env, ))

# Initialize the simulation
step(mx_model, mx_data)

cid = 0
cam = mx_model.cameras[cid]
cam_pose = cam.get_pose(mx_data)
link_poses = mx_model.get_link_poses(mx_data)

H, W = 240, 320
cfg = BatchSplatConfig(body_gaussians=gaussians, background_ply=None, minibatch=32)
renderer = MtxBatchSplatRenderer(cfg, mx_model)

# --- Step 2: batch_update_gaussians ---
body_pos = link_poses[...,:3]  # (Nenv, Nbody, 3)
body_quat = link_poses[...,3:]  # (Nenv, Nbody, 4) xyzw
assert np.linalg.norm(body_quat, axis=-1).min() > 1 - 1e-6 and np.linalg.norm(body_quat, axis=-1).max() < 1 + 1e-6, "Quaternion norm is too small."
print(body_pos.shape, body_quat.shape)

gsb = renderer.batch_update_gaussians(body_pos, body_quat)

In [ ]:
# --- Step 3: batch_env_render ---
cam_pos_lst = []
cam_xmat_lst = []
fovy_lst = []
for cid in range(len(mx_model.cameras)):
    cam = mx_model.cameras[cid]
    cam_pose = cam.get_pose(mx_data)
    cam_pos_lst.append(cam_pose[...,:3])
    cam_xmat_lst.append(Rotation.from_quat(cam_pose[... ,3:7]).as_matrix().reshape(num_env, 9))
    fovy_lst.append(mx_model.cameras[cid].fovy)
cam_pos = np.array(cam_pos_lst).transpose(1, 0, 2)
cam_xmat = np.array(cam_xmat_lst).transpose(1, 0, 2)
fovy = np.array(fovy_lst)
fovy = np.tile(fovy, (num_env, 1))

# 这个只在最开始运行一次即可，获取背景图像
bg_renderer = MtxBatchSplatRenderer(BatchSplatConfig(body_gaussians=dict(), background_ply=background), mx_model)
bg_gsb = bg_renderer.batch_update_gaussians(body_pos, body_quat)
bg_imgs, _ = bg_renderer.batch_env_render(bg_gsb, cam_pos, cam_xmat, H, W, fovy)

# --- Step 4: batch_env_render ---
rgb, depth = renderer.batch_env_render(gsb, cam_pos, cam_xmat, H, W, fovy, bg_imgs)

print('RGB:', rgb.shape, 'Depth:', depth.shape)
def tile(img, d):
    assert img.shape[0] == d*d
    img = img.reshape((d,d)+img.shape[1:])
    return np.concat(np.concat(img, axis=1), axis=1)
mediapy.show_image(tile(rgb[:16,0,...].cpu().numpy(), 4), width=1080)

In [ ]:
rgb.min()